In [2]:
!pip install pandas numpy scikit-learn torch

In [3]:
# importing libraries
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

Downloading the Dataset from Kaggle

In [4]:
# Download dataset
!kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/

Dataset URL: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/versions/
License(s): other
100% 25.7M/25.7M [00:00<00:00, 109MB/s]



In [5]:
import zipfile

zip_path = "imdb-dataset-of-50k-movie-reviews.zip"
extract_path = "/content/imdb"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extracted successfully!")

Extracted successfully!


In [6]:
# loading the dataset
df = pd.read_csv("/content/imdb/IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [7]:
df.shape

(50000, 2)

In [8]:
df['sentiment'].value_counts() # balanced dataset

,count
sentiment,
positive,25000
negative,25000


In [9]:
# Preprocessing
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

X_train, X_test, y_train, y_test = train_test_split(
    df['review'], df['sentiment'], test_size=0.2, random_state=42)

# Tokenization (pure Python, no Keras)
vocab_size = 10000
max_length = 200

# Build vocab from training data
from collections import Counter
import re

def tokenize(text):
    return re.findall(r'\w+', text.lower())

counter = Counter()
for review in X_train:
    counter.update(tokenize(review))

vocab = ['<PAD>', '<OOV>'] + [w for w, _ in counter.most_common(vocab_size - 2)]
word2idx = {w: i for i, w in enumerate(vocab)}

def encode_and_pad(texts, max_length):
    sequences = []
    for text in texts:
        tokens = [word2idx.get(w, 1) for w in tokenize(text)]  # 1 = <OOV>
        tokens = tokens[:max_length]
        tokens += [0] * (max_length - len(tokens))             # 0 = <PAD>
        sequences.append(tokens)
    return np.array(sequences)

X_train_pad = encode_and_pad(X_train, max_length)
X_test_pad  = encode_and_pad(X_test,  max_length)

# Convert to tensors
X_train_t = torch.tensor(X_train_pad, dtype=torch.long)
X_test_t  = torch.tensor(X_test_pad,  dtype=torch.long)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32)
y_test_t  = torch.tensor(y_test.values,  dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test_t,  y_test_t),  batch_size=64, shuffle=False)

In [10]:
df['sentiment'].value_counts() # for checking

,count
sentiment,
1,25000
0,25000


In [11]:
X_train.shape

(40000,)

In [12]:
# Models

class SentimentModel(nn.Module):
    def __init__(self, rnn_type='RNN'):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 128, padding_idx=0)

        if rnn_type == 'RNN':
            self.rnn = nn.RNN(128, 64, batch_first=True)
        elif rnn_type == 'LSTM':
            self.rnn = nn.LSTM(128, 64, batch_first=True)
        elif rnn_type == 'GRU':
            self.rnn = nn.GRU(128, 64, batch_first=True)

        self.rnn_type = rnn_type
        self.fc = nn.Linear(64, 1)

    def forward(self, x):
        x = self.embedding(x)
        if self.rnn_type == 'LSTM':
            out, (h, _) = self.rnn(x)
        else:
            out, h = self.rnn(x)
        return self.fc(h.squeeze(0))  # use last hidden state

In [13]:
# helper functions
def train_model(model, epochs=10):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters())

    for epoch in range(epochs):
        model.train()
        correct, total, running_loss = 0, 0, 0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            out = model(X_batch).squeeze(1)
            loss = criterion(out, y_batch)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            correct += ((out > 0) == y_batch.bool()).sum().item()
            total += len(y_batch)
        print(f"Epoch {epoch+1}: Loss={running_loss/len(train_loader):.4f}, Acc={correct/total:.4f}")

def evaluate_model(model):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            out = model(X_batch).squeeze(1)
            correct += ((out > 0) == y_batch.bool()).sum().item()
            total += len(y_batch)
    acc = correct / total
    print(f"Test Accuracy: {acc:.4f}")
    return acc

In [14]:
rnn_model  = SentimentModel('RNN');  train_model(rnn_model)
lstm_model = SentimentModel('LSTM'); train_model(lstm_model)
gru_model  = SentimentModel('GRU');  train_model(gru_model)

Epoch 1: Loss=0.6957, Acc=0.5078
Epoch 2: Loss=0.6865, Acc=0.5401
Epoch 3: Loss=0.6816, Acc=0.5414
Epoch 4: Loss=0.6653, Acc=0.5767
Epoch 5: Loss=0.6480, Acc=0.5929
Epoch 6: Loss=0.6254, Acc=0.6250
Epoch 7: Loss=0.6086, Acc=0.6283
Epoch 8: Loss=0.5973, Acc=0.6205
Epoch 9: Loss=0.5745, Acc=0.6505
Epoch 10: Loss=0.5548, Acc=0.6758
Epoch 1: Loss=0.6922, Acc=0.5178
Epoch 2: Loss=0.6908, Acc=0.5266
Epoch 3: Loss=0.6788, Acc=0.5585
Epoch 4: Loss=0.6390, Acc=0.6278
Epoch 5: Loss=0.5611, Acc=0.7175
Epoch 6: Loss=0.5836, Acc=0.6832
Epoch 7: Loss=0.5740, Acc=0.6854
Epoch 8: Loss=0.5473, Acc=0.6966
Epoch 9: Loss=0.4191, Acc=0.8264
Epoch 10: Loss=0.3928, Acc=0.8401
Epoch 1: Loss=0.6908, Acc=0.5213
Epoch 2: Loss=0.6732, Acc=0.5806
Epoch 3: Loss=0.6158, Acc=0.6763
Epoch 4: Loss=0.5979, Acc=0.6704
Epoch 5: Loss=0.5444, Acc=0.7139
Epoch 6: Loss=0.3518, Acc=0.8490
Epoch 7: Loss=0.2632, Acc=0.8951
Epoch 8: Loss=0.2103, Acc=0.9216
Epoch 9: Loss=0.1664, Acc=0.9408
Epoch 10: Loss=0.1262, Acc=0.9590


In [15]:
print(rnn_model)
print(lstm_model)
print(gru_model)

SentimentModel(
  (embedding): Embedding(10000, 128, padding_idx=0)
  (rnn): RNN(128, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)
SentimentModel(
  (embedding): Embedding(10000, 128, padding_idx=0)
  (rnn): LSTM(128, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)
SentimentModel(
  (embedding): Embedding(10000, 128, padding_idx=0)
  (rnn): GRU(128, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)


In [20]:
print("RNN  Accuracy:", evaluate_model(rnn_model))
print("LSTM Accuracy:", evaluate_model(lstm_model))
print("GRU  Accuracy:", evaluate_model(gru_model))

Test Accuracy: 0.5553
RNN  Accuracy: 0.5553
Test Accuracy: 0.8195
LSTM Accuracy: 0.8195
Test Accuracy: 0.8633
GRU  Accuracy: 0.8633


In [17]:
# Testing the fucntion
def predict(model, text):
    encoded = encode_and_pad([text], max_length)
    tensor = torch.tensor(encoded, dtype=torch.long)
    model.eval()
    with torch.no_grad():
        out = model(tensor).squeeze().item()
    return "Positive" if out > 0 else "Negative"

In [18]:
models = {'RNN' : rnn_model,
          'LSTM': lstm_model,
          'GRU': gru_model}

for name, model in models.items():
  print(f"{name}:")
  print(predict(model, "This movie was absolutely amazing!"))

RNN:
Positive
LSTM:
Negative
GRU:
Positive


In [19]:
models = {'RNN' : rnn_model,
          'LSTM': lstm_model,
          'GRU': gru_model}

for name, model in models.items():
  print(f"{name}:")
  print(predict(model, "Poor storyline, waste of money!"))

RNN:
Negative
LSTM:
Negative
GRU:
Negative
